# 06 — Evaluation

Full evaluation against FFIC ground truth: Precision@K, ROC-AUC, PR-AUC, LOCO test.

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path(".").resolve()))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve
sns_ok = False
try:
    import seaborn as sns; sns.set_theme(style="whitegrid"); sns_ok = True
except ImportError:
    pass

In [ ]:
from src.eval.metrics import evaluate_model, ensemble_scores, load_and_evaluate
if_scores = pd.read_parquet("data/processed/if_scores.parquet")
ae_scores = pd.read_parquet("data/processed/ae_scores.parquet")

In [ ]:
# Evaluate each model
if_metrics = evaluate_model(if_scores, "anomaly_score_if", name="Isolation Forest")
ae_metrics = evaluate_model(ae_scores, "anomaly_score_ae", name="Autoencoder")
ensemble = ensemble_scores(if_scores, ae_scores)
ens_metrics = evaluate_model(ensemble, "anomaly_score_ensemble", name="Ensemble (IF + AE)")

In [ ]:
# ROC curves
fig, ax = plt.subplots()
y_true = if_scores["is_leaked_market"]
for scores, label, c in [
    (if_scores["anomaly_score_if"], "IF", "steelblue"),
    (ae_scores.set_index("taker").loc[if_scores["taker"].values, "anomaly_score_ae"].values, "AE", "darkorange"),
    (ensemble["anomaly_score_ensemble"], "Ensemble", "crimson"),
]:
    try:
        fpr, tpr, _ = roc_curve(y_true, scores)
        from sklearn.metrics import auc
        ax.plot(fpr, tpr, label=f"{label} (AUC={auc(fpr,tpr):.3f})", color=c)
    except Exception as e:
        print(e)
ax.plot([0,1],[0,1],"k--",alpha=0.3)
ax.set_xlabel("FPR")
ax.set_ylabel("TPR")
ax.set_title("ROC Curves")
ax.legend()
plt.tight_layout()
plt.savefig("notebooks/fig_roc_curves.png", dpi=150)
plt.show()

In [ ]:
# Precision-Recall curves
fig, ax = plt.subplots()
for scores, label, c in [
    (if_scores["anomaly_score_if"], "IF", "steelblue"),
    (ae_scores.set_index("taker").loc[if_scores["taker"].values, "anomaly_score_ae"].values, "AE", "darkorange"),
    (ensemble["anomaly_score_ensemble"], "Ensemble", "crimson"),
]:
    try:
        prec, rec, _ = precision_recall_curve(y_true, scores)
        ax.plot(rec, prec, label=label, color=c)
    except Exception as e:
        print(e)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves")
ax.legend()
plt.tight_layout()
plt.savefig("notebooks/fig_pr_curves.png", dpi=150)
plt.show()

In [ ]:
# LOCO test with Isolation Forest
from src.eval.metrics import leave_one_case_out
from src.models.isolation_forest import run as run_if
profiles = pd.read_parquet("data/processed/trader_profiles.parquet")

loco_results = leave_one_case_out(
    profiles,
    train_fn=lambda p: run_if(profiles=p, save=False),
    score_col="anomaly_score_if",
)
print("
LOCO Results:")
print(loco_results.to_string(index=False))
print(f"
Mean Precision@20: {loco_results['precision_at_20'].mean():.3f}")
print(f"Mean ROC-AUC:      {loco_results['roc_auc'].mean():.3f}")

In [ ]:
# Summary table
results = pd.DataFrame([if_metrics, ae_metrics, ens_metrics])
print("
Summary Table:")
print(results.to_string(index=False))